# Agents and Vector Databases - Solution (Milvus Lite + Web Scraping)

This notebook builds a local Milvus Lite vector database from scraped web pages, then queries it and uses a simple tool-calling agent.

Instructor note: this solution includes the full vector DB creation step (no TODOs).

In [ ]:
# Install dependencies (run once)
!pip -q install "pymilvus[milvus-lite]" sentence-transformers beautifulsoup4 lxml requests openai

In [ ]:
from pathlib import Path
import json
import re
import time
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient

BASE_DIR = Path(".")
MILVUS_DB_PATH = BASE_DIR / "milvus_kb.db"
COLLECTION_NAME = "web_kb"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"
}

np.random.seed(7)

In [ ]:
SOURCE_URLS = [
    {"topic": "OpenAI", "url": "https://en.wikipedia.org/wiki/OpenAI"},
    {"topic": "NVIDIA", "url": "https://en.wikipedia.org/wiki/Nvidia"},
    {"topic": "Tesla", "url": "https://en.wikipedia.org/wiki/Tesla,_Inc."},
    {"topic": "ASML", "url": "https://en.wikipedia.org/wiki/ASML_Holding"},
    {"topic": "SAP", "url": "https://en.wikipedia.org/wiki/SAP"},
    {"topic": "Siemens", "url": "https://en.wikipedia.org/wiki/Siemens"},
    {"topic": "Spotify", "url": "https://en.wikipedia.org/wiki/Spotify"},
    {"topic": "IKEA", "url": "https://en.wikipedia.org/wiki/IKEA"},
    {"topic": "Zara", "url": "https://en.wikipedia.org/wiki/Zara_(retailer)"},
    {"topic": "DHL", "url": "https://en.wikipedia.org/wiki/DHL"},
    {"topic": "GDPR", "url": "https://en.wikipedia.org/wiki/General_Data_Protection_Regulation"},
    {"topic": "EU AI Act", "url": "https://en.wikipedia.org/wiki/Artificial_Intelligence_Act"},
]


def fetch_wikipedia_page(url, max_paragraphs=12):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else url

    content = soup.select_one("#mw-content-text")
    if not content:
        return {"title": title, "url": url, "text": ""}

    paragraphs = []
    for p in content.find_all("p", recursive=True):
        text = p.get_text(" ", strip=True)
        text = re.sub(r"\[\d+\]", "", text)
        if len(text) >= 80:
            paragraphs.append(text)
        if len(paragraphs) >= max_paragraphs:
            break

    return {
        "title": title,
        "url": url,
        "text": "\n".join(paragraphs),
    }


def chunk_text(text, max_chars=900):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = []
    current_len = 0

    for p in paragraphs:
        if current_len + len(p) + 1 > max_chars and current:
            chunks.append(" ".join(current))
            current = [p]
            current_len = len(p)
        else:
            current.append(p)
            current_len += len(p) + 1

    if current:
        chunks.append(" ".join(current))

    return chunks

## Step 1: Scrape and chunk the dataset

In [ ]:
pages = []
for item in SOURCE_URLS:
    try:
        page = fetch_wikipedia_page(item["url"])
        page["topic"] = item["topic"]
        pages.append(page)
        time.sleep(0.5)
    except Exception as exc:
        print(f"Failed to fetch {item['url']}: {exc}")

chunks = []
chunk_id = 0
for page in pages:
    for i, chunk in enumerate(chunk_text(page["text"], max_chars=900)):
        chunks.append(
            {
                "id": chunk_id,
                "title": page["title"],
                "topic": page["topic"],
                "url": page["url"],
                "chunk_id": i,
                "text": chunk,
            }
        )
        chunk_id += 1

if not chunks:
    raise RuntimeError("No content scraped. Check network access.")

df_docs = pd.DataFrame(chunks)
print("Pages:", len(pages))
print("Chunks:", len(df_docs))
print(df_docs[["id", "title", "chunk_id", "url"]].head())

## Step 2: Create the Milvus Lite vector database

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
texts = df_docs["text"].tolist()
embeddings = model.encode(texts, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype="float32")

client = MilvusClient(str(MILVUS_DB_PATH))

if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=embeddings.shape[1],
    metric_type="IP",
    enable_dynamic_field=True,
)

records = []
for row, vector in zip(df_docs.to_dict(orient="records"), embeddings):
    records.append(
        {
            "id": int(row["id"]),
            "vector": vector.tolist(),
            "title": row["title"],
            "topic": row["topic"],
            "url": row["url"],
            "chunk_id": int(row["chunk_id"]),
            "text": row["text"],
        }
    )

client.insert(collection_name=COLLECTION_NAME, data=records)
client.load_collection(collection_name=COLLECTION_NAME)

search_params = {"metric_type": "IP", "params": {}}


def vector_search(query, top_k=3):
    query_vec = model.encode([query], normalize_embeddings=True)
    query_vec = np.array(query_vec, dtype="float32")

    results = client.search(
        collection_name=COLLECTION_NAME,
        data=query_vec.tolist(),
        limit=top_k,
        output_fields=["title", "topic", "url", "chunk_id", "text"],
        search_params=search_params,
    )

    formatted = []
    for rank, hit in enumerate(results[0], start=1):
        entity = hit.get("entity", {})
        formatted.append(
            {
                "rank": rank,
                "score": float(hit.get("distance", 0.0)),
                "id": hit.get("id"),
                "title": entity.get("title"),
                "topic": entity.get("topic"),
                "url": entity.get("url"),
                "chunk_id": entity.get("chunk_id"),
                "text": entity.get("text"),
            }
        )

    return formatted

## Step 3: Retrieval and tool-calling agent

In [ ]:
example_queries = [
    "Which company is headquartered in Eindhoven and makes lithography systems?",
    "Which law focuses on data privacy for EU citizens?",
    "Which company started as a music streaming service in Sweden?",
    "Which firm is known for its enterprise resource planning software?",
]

for q in example_queries:
    print("\nQuestion:", q)
    results = vector_search(q, top_k=3)
    for r in results:
        print(
            f"- {r['title']} (chunk {r['chunk_id']}) score={r['score']:.3f}"
        )

# --- ReAct-style agent (tool calling) ---
import os
from openai import OpenAI

OPENAI_MODEL = "gpt-4o-mini"
client = None
if os.getenv("OPENAI_API_KEY"):
    client = OpenAI()

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "vector_search",
            "description": "Search the scraped knowledge base for relevant chunks.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "top_k": {"type": "integer", "default": 3},
                },
                "required": ["query"],
            },
        },
    }
]

SYSTEM_PROMPT = (
    "You are a concise business analyst. Use vector_search when needed. "
    "Answer in 2-4 bullet points and cite title and URL for each claim."
)


def run_agent(question, max_steps=3):
    if client is None:
        results = vector_search(question, top_k=3)
        lines = ["No OPENAI_API_KEY set. Showing top retrieval results only:"]
        for r in results:
            lines.append(f"- {r['title']} ({r['url']})")
        return "\n".join(lines)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=TOOLS,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for call in msg.tool_calls:
                if call.function.name == "vector_search":
                    args = json.loads(call.function.arguments)
                    results = vector_search(args["query"], args.get("top_k", 3))
                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": call.id,
                            "name": "vector_search",
                            "content": json.dumps(results),
                        }
                    )
        else:
            return msg.content

    return "Agent stopped without a final answer."

# Try:
# print(run_agent("Summarize the business models of SAP and Siemens."))
# print(run_agent("Which law and which company are most relevant to EU AI governance?"))